# 18장 · 도구 요청의 인자 검증과 실행 차단

도구를 실행하기 전에 이름과 호출 ID, 인자의 형식을 검사하면 잘못된 요청이 어느 조건에서 막혔는지 기록할 수 있습니다.

대출 규칙 조회 요청의 함수 이름, 인자, 호출 ID를 실행 전에 검사합니다. 유효하지 않은 요청과 정상 호출 뒤 빈 결과가 나온 경우를 구분합니다.

실행을 확인한 커널은 Python 3.12.15입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.

## JSON 객체와 빈 호출 ID 구분

get_loan_rule은 입력 칸이 없는 도구입니다. 요청에는 이름과 비어 있지 않은 호출 ID가 있어야 하고 arguments에는 빈 JSON 객체를 적은 문자열이 와야 합니다.

### 셀 01 · arguments의 빈 JSON 객체

겉보기에는 내용이 없어도 문자열을 풀었을 때 어떤 값이 남는지가 중요합니다.

In [1]:
import json
item = {"name": "get_loan_rule", "call_id": "c1", "arguments": "{}"}
print(json.loads(item["arguments"]) == {})

True


True가 나옵니다. 빈 문자열은 JSON 문서가 아니어서 같은 변환을 통과할 수 없습니다. []는 읽을 수 있는 JSON이지만 이 도구가 정한 객체 모양과 다릅니다.

도구에 건넬 인자가 하나도 없더라도 '{}'는 읽을 수 있는 JSON 객체입니다. 반면 call_id가 비어 있으면 실행 뒤 결과를 되돌려 줄 대상을 특정할 수 없습니다. 두 값 모두 화면에서는 ‘빈 값’처럼 보여도 검사하는 지점이 다릅니다.

### 셀 02 · strip()으로 빈 call_id 검사

ID에 공백 세 칸만 들어온 경우를 따로 확인합니다.

In [2]:
call_id = "   "
print(bool(call_id.strip()))

False


False이므로 결과를 연결할 이름이 없습니다. strip을 빼면 원래 세 칸의 길이 때문에 유효하다고 잘못 판단할 수 있습니다.

### 빈 인자 객체와 빈 호출 ID

<p>get_loan_rule은 별도로 받을 인자가 없어 문자열 '{}'를 풀면 빈 딕셔너리가 됩니다. 반면 공백만 있는 call_id는 strip()한 뒤 비어 있으므로 나중에 결과와 짝지을 수 없습니다.</p>

## validate_request()의 필드별 검사

요청의 바깥 자료형부터 확인하고 안쪽 칸을 읽습니다. 유효하지 않은 입력이면 함수 안에서 False를 돌려주면 됩니다. 사전의 값을 덮어쓰지 않으면 실패 원인을 나중에도 다시 읽을 수 있습니다.

### 검사 함수의 입력·출력 계약

<p>validate_request는 호출 요청을 검사해 bool을 반환합니다. 딕셔너리가 아닌 값이나 잘못된 요청에는 False를 반환합니다. 실제 도구나 모델을 실행하지 않고 입력도 수정하지 않습니다.</p>

<p>허용 이름은 get_loan_rule 하나입니다. call_id는 공백이 아닌 문자열, arguments는 JSON 문자열이며 파싱 결과가 빈 객체 {}여야 합니다. bytes나 이미 만들어진 딕셔너리는 JSON 문자열이 아니므로 거부합니다. 잘못된 JSON·추가 인자·누락된 필드에도 False를 반환합니다.</p>

<table><caption>입출력 예</caption><thead><tr><th scope="col">입력의 차이</th><th scope="col">반환</th></tr>
</thead><tbody><tr><td>허용 이름·c1·{}</td><td>True</td></tr>
<tr><td>다른 함수 이름</td><td>False</td></tr>
<tr><td>빈 call_id</td><td>False</td></tr>
<tr><td>arguments=[]</td><td>False</td></tr>
<tr><td>깨진 JSON</td><td>False</td></tr>
<tr><td>arguments를 풀었을 때 필드가 있는 객체</td><td>False</td></tr>
<tr><td>공백·비문자열 ID 또는 누락된 필드</td><td>False</td></tr>
<tr><td>None·리스트 입력</td><td>False</td></tr>
<tr><td>문자열이 아닌 arguments 또는 JSON null</td><td>False</td></tr>
</tbody></table>

#### call_id가 같은 요청의 필드별 오류

<table style="width:100%;border-collapse:collapse"><thead><tr><th scope="col">원본에서 바꾼 부분</th><th scope="col">풀이 함수의 반환</th><th scope="col">이유</th></tr></thead><tbody><tr><td>원본 arguments는 문자열 &quot;{}&quot;</td><td>True</td><td>도구에 전달할 인자가 없음</td></tr><tr><td>arguments를 문자열 {&quot;day&quot;:&quot;weekend&quot;}로 변경</td><td>False</td><td>JSON을 푼 안쪽 객체에 인자가 생김</td></tr><tr><td>바깥 item에만 day를 추가하고 arguments는 &quot;{}&quot; 유지</td><td>True</td><td>풀이 함수는 바깥의 부가 필드를 읽지 않음</td></tr></tbody></table>

### 과제 03 · validate_request() 완성

사전, 도구 이름, 공백이 아닌 문자열 ID, 문자열로 온 빈 JSON 객체 순서로 검사하세요. 변환에 실패해도 예외 대신 False를 돌려줍니다.

get_loan_rule은 인자를 받지 않습니다. 요청 바깥에서 name과 call_id를 읽고, arguments 문자열을 풀어 빈 객체인지 확인하세요. day를 어느 쪽에 넣었는지에 따라 검사 결과가 달라집니다.

In [3]:
def validate_request(item):
    pass  # 위 조건을 차례로 검사하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>def validate_request(item):
    if not isinstance(item, dict) or item.get(&quot;name&quot;) != &quot;get_loan_rule&quot;:
        return False
    cid = item.get(&quot;call_id&quot;)
    if not isinstance(cid, str) or not cid.strip():
        return False
    raw = item.get(&quot;arguments&quot;)
    if not isinstance(raw, str):
        return False
    try:
        return json.loads(raw) == {}
    except ValueError:
        return False

print(validate_request(item))</code></pre>
<p>실행 결과</p><pre><code>True</code></pre>
<p>원래 c1 요청은 True입니다. 먼저 자료형을 검사해 None이나 목록이 들어와도 .get을 잘못 호출하는 오류를 피할 수 있습니다. 문자열로 온 {}만 허용하므로 파이썬 사전 자체를 넣은 요청도 거부합니다.</p>
</details>

### 요청 필드의 검증 순서

<p>처음부터 한 줄로 모든 조건을 쓰지 말고 이름, ID, JSON 파싱, 객체 내용의 순서로 나누세요. JSON 읽기가 실패하는 경우는 try/except로 처리합니다. 파싱이 성공해도 리스트나 숫자이면 객체 조건을 통과하지 못해야 합니다.</p>

<p>각 조건을 추가할 때 하나의 정상 사례와 하나의 실패 사례를 직접 호출해 봅니다. True와 "True"는 다릅니다. 이 함수는 문자열이 아닌 bool을 반환해야 합니다.</p>

## 허용 도구와 거부 원인 기록

이제 증상을 보고 필요한 부분으로 돌아갑니다. 인자를 풀지 못한 경우라면 15장의 arguments를, 요청 ID와 결과 ID가 다르면 16장 왕복을, 상한이 보이면 17장 마지막 반복을 다시 살펴보세요.

### 원본 요청의 필드를 하나씩 바꿔 검사

<p>원본 item에서 도구 이름, 호출 ID, arguments 중 하나만 바꾸어 실행하세요. 어떤 필드가 True를 False로 바꾸었는지 바로 알아볼 수 있습니다. 특히 문자열 'null'과 '{}'를 각각 넣었을 때의 차이를 확인하세요.</p>

### 셀 04 · 허용하지 않은 도구 이름 판별

도구 이름이 허용 목록과 다른 요청을 받았습니다. 프로그램이 기록할 값 한 개를 계산합니다.

In [4]:
bad = {**item, "name": "other_lookup"}
allowed = bad["name"] == "get_loan_rule"
print(allowed)

False


False는 도구 선택 검사 결과입니다. 대출 기간이 0일이거나 데이터가 비어 있다는 뜻으로 바꿔 적으면 안 됩니다.

### 검증 결과에서 실행 여부 결정하기

<p>bad에서는 call_id가 c1이지만 name이 other_lookup입니다. 허용된 이름과 다르면 도구 함수를 호출하지 않습니다. 이후 진단에는 이 요청의 ID와 받은 이름을 함께 남깁니다.</p>

### 과제 05 · call_id와 name으로 실패 기록

요청 ID c1과 실패한 필드 이름을 한 문장으로 연결하세요. 실제 실행 결과를 만들지 마세요.

In [5]:
# bad의 call_id와 name을 읽어 진단 문구를 작성하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>print(f&quot;{bad[&#x27;call_id&#x27;]} 요청은 도구 이름 {bad[&#x27;name&#x27;]}이 허용 목록에 없어 실행 전에 멈췄습니다.&quot;)</code></pre>
<p>실행 결과</p><pre><code>c1 요청은 도구 이름 other_lookup이 허용 목록에 없어 실행 전에 멈췄습니다.</code></pre>
<p>다른 사람이 c1의 원본 요청으로 돌아가 name을 고칠 수 있습니다. 이름을 고친 뒤에는 arguments와 call_id도 다시 확인하세요.</p>
</details>

### 재현 가능한 실패 로그

<p>실패 기록의 c1은 원래 요청으로 돌아갈 단서입니다. 실패 이유는 허용 목록에 없는 이름입니다. 원래 item을 덮어쓰지 않고 bad를 만들었으므로 두 요청을 나란히 놓고 name을 비교할 수 있습니다.</p>

## 돌아보기

입력 오류를 찾았으면 검증 결과와 실제 도구 호출 횟수를 함께 확인하세요. 아래 실습에서는 허용하지 않은 인자 때문에 실행 함수가 한 번도 불리지 않는 경우를 재현합니다.

### 과제 06 · 실행할 요청만 골라내는 입구 함수

과제 03의 validate_request를 완성한 뒤, 검증에 통과한 요청만 실행 함수로 넘기는 dispatch를 작성하세요. get_loan_rule에 인자 배열을 보낸 경우 실행 횟수도 확인합니다.

In [ ]:
from unittest.mock import Mock
lookup = Mock(return_value=14)
request = {**item, "arguments": "[]"}
# dispatch(request, lookup)의 반환값과 lookup.call_count를 출력하세요.

<details><summary>풀이와 실행 결과 보기</summary>
<pre><code>from unittest.mock import Mock
lookup = Mock(return_value=14)
request = {**item, "arguments": "[]"}

def dispatch(request, lookup):
    if not validate_request(request):
        return "거부"
    return lookup()

print(dispatch(request, lookup), lookup.call_count)</code></pre>
<p>실행 결과</p><pre><code>거부 0</code></pre>
<p>[]는 파싱 가능한 JSON이지만 이 도구가 허용하는 빈 객체와 다릅니다. 모의 함수의 호출 횟수 0이 검사 이후 실제로 실행을 건너뛰었다는 증거입니다.</p>
</details>